In [ ]:
# Project C - From Raw to EU-27
Research Question: After 20 years of cohesion funding, is the gap between rich and poor NUTS-2 regions closing or widening on income?

Steps:
1. Inspect raw Eurostat files
2. Filter to the right indicator (unit, sex, age)
3. Keep NUTS-2 only (geo = 4 chars) + EU-27 only
4. Save as separate Project_*_EU27.csv files

In [11]:
import pandas as pd

raw_files = ["raw_pop.csv", "raw_emp.csv", "raw_youth.csv",
             "raw_gdp.csv", "raw_gva.csv"]

for f in raw_files:
    df = pd.read_csv(f, nrows=5)
    print("\n===", f, "===")
    print("Columns:", df.columns.tolist())

    # Full file, just to see filter values
    full = pd.read_csv(f, usecols=lambda c: c not in ["TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"])
    for col in full.columns:
        if col not in ["freq", "geo"]:
            print(f"  {col}: {sorted(full[col].dropna().unique().astype(str))[:10]}")


=== raw_pop.csv ===
Columns: ['freq', 'unit', 'sex', 'age', 'geo', 'time', 'value']
  unit: ['NR']
  sex: ['T']
  age: ['TOTAL']
  time: [np.str_('1990'), np.str_('1991'), np.str_('1992'), np.str_('1993'), np.str_('1994'), np.str_('1995'), np.str_('1996'), np.str_('1997'), np.str_('1998'), np.str_('1999')]
  value: [np.str_('0.0'), np.str_('1000142.0'), np.str_('10001843.0'), np.str_('1000348.0'), np.str_('10005372.0'), np.str_('1000684.0'), np.str_('1000696.0'), np.str_('10008242.0'), np.str_('10008659.0'), np.str_('10010833.0')]

=== raw_emp.csv ===
Columns: ['freq', 'unit', 'sex', 'age', 'geo', 'time', 'value']
  unit: ['PC']
  sex: ['T']
  age: ['Y20-64']
  time: [np.str_('1999'), np.str_('2000'), np.str_('2001'), np.str_('2002'), np.str_('2003'), np.str_('2004'), np.str_('2005'), np.str_('2006'), np.str_('2007'), np.str_('2008')]
  value: [np.str_('30.1'), np.str_('30.7'), np.str_('31.5'), np.str_('31.7'), np.str_('31.8'), np.str_('32.2'), np.str_('32.7'), np.str_('32.8'), np.str

In [12]:
# Cell 3 - Clean raw -> Project_*_EU27.csv
# Raw -> filter indicator -> NUTS-2 only -> EU-27 only -> 2000-2024 only -> save separate
# GVA is converted to per capita using population at the end

import pandas as pd

EU27 = ["AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR",
        "DE","EL","HU","IE","IT","LV","LT","LU","MT","NL",
        "PL","PT","RO","SK","SI","ES","SE"]

FILTERS = {
    "raw_pop.csv":   {"unit": "NR",      "sex": "T", "age": "TOTAL"},
    "raw_emp.csv":   {"unit": "PC",      "sex": "T", "age": "Y20-64"},
    "raw_youth.csv": {"unit": "PC",      "sex": "T", "age": "Y15-24"},
    # FIXED - real units from diagnostic
    "raw_gdp.csv":   {"unit": "PPS_EU27_2020_HAB"},
    "raw_gva.csv":   {"unit": "CP_MEUR", "nace_r2": "TOTAL"},
}

OUTPUT = {
    "raw_pop.csv":   ("Project_pop_EU27.csv", "pop"),
    "raw_emp.csv":   ("Project_emp_EU27.csv", "emp_rate"),
    "raw_youth.csv": ("Project_youth_EU27.csv", "youth_unemp"),
    "raw_gdp.csv":   ("Project_gdp_EU27.csv", "gdp_pc"),
    # GVA saved as total first, converted below
    "raw_gva.csv":   ("Project_gva_EU27.csv", "gva_total"),
}

def clean_raw(path, filters, out_path, value_name):
    df = pd.read_csv(path)
    print(f"\n{path}: start {df.shape}")

    df = df.rename(columns={"TIME_PERIOD": "year", "time": "year",
                            "OBS_VALUE": "value"})
    df["geo"] = df["geo"].astype(str).str.strip().str.upper()
    df["year"] = df["year"].astype(int)

    for col, val in filters.items():
        if col in df.columns:
            df = df[df[col] == val]
            print(f"  {col} == {val}: {len(df)} rows left")

    df = df[df["geo"].str.len() == 4]
    df = df[df["geo"].str[:2].isin(EU27)]
    df = df[(df["year"] >= 2000) & (df["year"] <= 2024)]

    df = df.dropna(subset=["value"])
    df = df.rename(columns={"value": value_name})
    df = df.drop_duplicates(subset=["geo", "year"])
    df = df[["geo", "year", value_name]].sort_values(["geo", "year"])

    df.to_csv(out_path, index=False)
    print(f"  -> {out_path}: {df['geo'].nunique()} regions, "
          f"{df['year'].min()}-{df['year'].max()}, {len(df)} rows")
    return df

for raw, filters in FILTERS.items():
    out_path, value_name = OUTPUT[raw]
    clean_raw(raw, filters, out_path, value_name)

# --- Convert GVA total -> GVA per capita ---
# raw_gva is CP_MEUR = million euro total, so: gva_pc = gva_total * 1,000,000 / pop
pop = pd.read_csv("Project_pop_EU27.csv")
gva = pd.read_csv("Project_gva_EU27.csv")

gva_pc = gva.merge(pop, on=["geo", "year"], how="inner")
gva_pc["gva_pc"] = gva_pc["gva_total"] * 1_000_000 / gva_pc["pop"]
gva_pc = gva_pc[["geo", "year", "gva_pc"]].sort_values(["geo", "year"])
gva_pc.to_csv("Project_gva_EU27.csv", index=False)
print(f"\nGVA converted to per capita: {gva_pc['geo'].nunique()} regions, "
      f"{gva_pc['year'].min()}-{gva_pc['year'].max()}, {len(gva_pc)} rows")

print("\nDone - 5 separate Project_*_EU27.csv files, EU-27, 2000-2024 only.")
print("GDP = PPS per capita (already per capita in Eurostat)")
print("GVA = CP per capita (converted from total / population)")


raw_pop.csv: start (19152, 7)
  unit == NR: 19152 rows left
  sex == T: 19152 rows left
  age == TOTAL: 19152 rows left
  -> Project_pop_EU27.csv: 261 regions, 2000-2024, 5847 rows

raw_emp.csv: start (13824, 7)
  unit == PC: 13824 rows left
  sex == T: 13824 rows left
  age == Y20-64: 13824 rows left
  -> Project_emp_EU27.csv: 256 regions, 2000-2024, 5839 rows

raw_youth.csv: start (82944, 8)
  unit == PC: 82944 rows left
  sex == T: 82944 rows left
  age == Y15-24: 82944 rows left
  -> Project_youth_EU27.csv: 252 regions, 2000-2024, 5003 rows

raw_gdp.csv: start (11775, 5)
  unit == PPS_EU27_2020_HAB: 11775 rows left
  -> Project_gdp_EU27.csv: 244 regions, 2000-2024, 5973 rows

raw_gva.csv: start (680625, 6)
  unit == CP_MEUR: 680625 rows left
  nace_r2 == TOTAL: 45375 rows left
  -> Project_gva_EU27.csv: 258 regions, 2000-2024, 6414 rows

GVA converted to per capita: 244 regions, 2000-2024, 5642 rows

Done - 5 separate Project_*_EU27.csv files, EU-27, 2000-2024 only.
GDP = PPS per 

In [16]:
# Sector GVA: 6 broad sectors, shares of TOTAL
# A, B-E, F, G-J, K-N, O-U -> these 6 sum to TOTAL, no double-counting
# Source: raw_gva.csv, unit = CP_MEUR

import pandas as pd

EU27 = ["AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR",
        "DE","EL","HU","IE","IT","LV","LT","LU","MT","NL",
        "PL","PT","RO","SK","SI","ES","SE"]

# Mapping if file only has single letters A, B, C, D...
LETTER_TO_GROUP = {
    "A": "A",
    "B": "B-E", "C": "B-E", "D": "B-E", "E": "B-E",
    "F": "F",
    "G": "G-J", "H": "G-J", "I": "G-J", "J": "G-J",
    "K": "K-N", "L": "K-N", "M": "K-N", "N": "K-N",
    "O": "O-U", "P": "O-U", "Q": "O-U", "R": "O-U",
    "S": "O-U", "T": "O-U", "U": "O-U",
}

GROUPS = ["A", "B-E", "F", "G-J", "K-N", "O-U"]

df = pd.read_csv("raw_gva.csv")
print("Columns:", df.columns.tolist())
print("nace_r2 values in file:", sorted(df["nace_r2"].dropna().unique().astype(str)))
print("unit values:", sorted(df["unit"].dropna().unique().astype(str)))

df = df.rename(columns={"time": "year", "TIME_PERIOD": "year", "OBS_VALUE": "value"})
df["geo"] = df["geo"].astype(str).str.strip().str.upper()
df["year"] = df["year"].astype(int)

# 1. Only CP_MEUR, EU-27, NUTS-2, 2000-2024
df = df[df["unit"] == "CP_MEUR"]
df = df[df["geo"].str.len() == 4]
df = df[df["geo"].str[:2].isin(EU27)]
df = df[(df["year"] >= 2000) & (df["year"] <= 2024)]
df = df.dropna(subset=["value"])

# 2. Get TOTAL separately - denominator for shares
total = df[df["nace_r2"] == "TOTAL"][["geo", "year", "value"]].rename(columns={"value": "gva_total"})
total = total.drop_duplicates(subset=["geo", "year"])

# 3. Build the 6 groups
# Case 1: file already has aggregates B-E, G-J, K-N, O-U -> use them directly
if set(GROUPS).issubset(set(df["nace_r2"].unique())):
    print("\nFile already has aggregates B-E, G-J, K-N, O-U - using them directly")
    sectors = df[df["nace_r2"].isin(GROUPS)][["geo", "year", "nace_r2", "value"]]
    sectors = sectors.rename(columns={"nace_r2": "sector", "value": "gva"})
# Case 2: file only has single letters -> sum them into groups
else:
    print("\nFile has single letters - summing into 6 groups")
    df2 = df[df["nace_r2"].isin(LETTER_TO_GROUP.keys())].copy()
    df2["sector"] = df2["nace_r2"].map(LETTER_TO_GROUP)
    sectors = df2.groupby(["geo", "year", "sector"], as_index=False)["value"].sum()
    sectors = sectors.rename(columns={"value": "gva"})

sectors = sectors.drop_duplicates(subset=["geo", "year", "sector"])

# 4. FIX: Keep only geo-year with ALL 6 sectors
# Otherwise shares sum to <100% or 0% if a sector is missing
counts = sectors.groupby(["geo", "year"]).size().reset_index(name="n_sectors")
print(f"\nGeo-years before fix: {len(counts)}")
print("Sectors per geo-year:")
print(counts["n_sectors"].value_counts().sort_index())

complete_keys = counts[counts["n_sectors"] == 6][["geo", "year"]]
sectors = sectors.merge(complete_keys, on=["geo", "year"], how="inner")
total = total.merge(complete_keys, on=["geo", "year"], how="inner")
print(f"Geo-years after keeping complete only: {len(complete_keys)}")
print(f"Regions after fix: {sectors['geo'].nunique()}")

# 5. Share = sector / TOTAL * 100
out = sectors.merge(total, on=["geo", "year"], how="inner")
out["share_pct"] = out["gva"] / out["gva_total"] * 100
out = out[["geo", "year", "sector", "gva", "gva_total", "share_pct"]].sort_values(["geo", "year", "sector"])

out.to_csv("Project_gva_sector_EU27.csv", index=False)
print(f"\n-> Project_gva_sector_EU27.csv: {out['geo'].nunique()} regions, "
      f"{out['year'].min()}-{out['year'].max()}, {len(out)} rows")

# 6. Check: 6 shares should sum to ~100% for each geo-year
check = out.groupby(["geo", "year"])["share_pct"].sum().describe()
print("\nSum of 6 shares per geo-year - should be ~100:")
print(check[["mean", "min", "max"]])
print("\nExample - one region, 2024:")
print(out[(out["geo"] == out["geo"].iloc[0]) & (out["year"] == 2024)].to_string(index=False))

Columns: ['freq', 'unit', 'nace_r2', 'geo', 'time', 'value']
nace_r2 values in file: ['A', 'B-E', 'C', 'F', 'G-I', 'G-J', 'J', 'K', 'K-N', 'L', 'M_N', 'O-Q', 'O-U', 'R-U', 'TOTAL']
unit values: ['CP_MEUR']

File already has aggregates B-E, G-J, K-N, O-U - using them directly

Geo-years before fix: 6346
Sectors per geo-year:
n_sectors
1     122
2      28
4       1
6    6195
Name: count, dtype: int64
Geo-years after keeping complete only: 6195
Regions after fix: 252

-> Project_gva_sector_EU27.csv: 252 regions, 2000-2024, 37170 rows

Sum of 6 shares per geo-year - should be ~100:
mean     99.596475
min       0.000000
max     100.035769
Name: share_pct, dtype: float64

Example - one region, 2024:
 geo  year sector    gva  gva_total  share_pct
AT11  2024      A  308.0    10559.0   2.916943
AT11  2024    B-E 1791.0    10559.0  16.961834
AT11  2024      F 1001.0    10559.0   9.480064
AT11  2024    G-J 2212.0    10559.0  20.948953
AT11  2024    K-N 2475.0    10559.0  23.439720
AT11  2024    O

In [19]:
# Fix: Drop extra-regio ZZ regions (HUZZ, DEZZ, etc.) - not real NUTS-2 regions
import pandas as pd

out = pd.read_csv("Project_gva_sector_EU27.csv")

print(f"Before: {out['geo'].nunique()} regions, {len(out)} rows")
print("ZZ regions in file:", sorted(out[out['geo'].str.endswith('ZZ')]['geo'].unique()))

# Drop any geo ending in ZZ
out = out[~out['geo'].str.endswith('ZZ')]

out.to_csv("Project_gva_sector_EU27.csv", index=False)
print(f"After: {out['geo'].nunique()} regions, {len(out)} rows")

# Re-check
check = out.groupby(["geo", "year"])["share_pct"].sum().describe()
print("\nSum of 6 shares per geo-year - should be ~100:")
print(check[["mean", "min", "max"]])

bad = out.groupby(["geo", "year"])["share_pct"].sum().reset_index(name="sum_share")
bad = bad[(bad["sum_share"] < 99) | (bad["sum_share"] > 101)]
print(f"\nBad geo-years left: {len(bad)}")

Before: 252 regions, 37170 rows
ZZ regions in file: ['ATZZ', 'DKZZ', 'ESZZ', 'FRZZ', 'HUZZ', 'ITZZ', 'MTZZ', 'NLZZ']
After: 244 regions, 36000 rows

Sum of 6 shares per geo-year - should be ~100:
mean    100.000028
min      99.980092
max     100.035769
Name: share_pct, dtype: float64

Bad geo-years left: 0


In [21]:
import pandas as pd, glob

for f in sorted(glob.glob("Project_*_EU27.csv")):
    df = pd.read_csv(f)
    zz = sorted(df[df['geo'].astype(str).str.endswith('ZZ')]['geo'].unique())
    print(f, "-> ZZ:", zz, "| regions:", df['geo'].nunique())

Project_emp_EU27.csv -> ZZ: [] | regions: 256
Project_gdp_EU27.csv -> ZZ: [] | regions: 244
Project_gva_EU27.csv -> ZZ: [] | regions: 244
Project_gva_sector_EU27.csv -> ZZ: [] | regions: 244
Project_pop_EU27.csv -> ZZ: [] | regions: 261
Project_youth_EU27.csv -> ZZ: [] | regions: 252


In [22]:
# Cell 4 - Verify final files
import pandas as pd

for f in ["Project_pop_EU27.csv", "Project_emp_EU27.csv", "Project_youth_EU27.csv",
          "Project_gdp_EU27.csv", "Project_gva_EU27.csv"]:
    df = pd.read_csv(f)
    print(f, df.shape,
          "| geo len 4:", (df["geo"].str.len() == 4).all(),
          "| missing:", df.isna().sum().sum(),
          "| regions:", df["geo"].nunique(),
          "| years:", df["year"].min(), "-", df["year"].max())

Project_pop_EU27.csv (5847, 3) | geo len 4: True | missing: 0 | regions: 261 | years: 2000 - 2024
Project_emp_EU27.csv (5839, 3) | geo len 4: True | missing: 0 | regions: 256 | years: 2000 - 2024
Project_youth_EU27.csv (5003, 3) | geo len 4: True | missing: 0 | regions: 252 | years: 2000 - 2024
Project_gdp_EU27.csv (5973, 3) | geo len 4: True | missing: 0 | regions: 244 | years: 2000 - 2024
Project_gva_EU27.csv (5642, 3) | geo len 4: True | missing: 0 | regions: 244 | years: 2000 - 2024


In [23]:
# Final sample for MAIN analysis - GDP PPS only
import pandas as pd

gdp = pd.read_csv("Project_gdp_EU27.csv")

print("GDP PPS regions in file:", gdp["geo"].nunique())

pivot = gdp[gdp["year"].isin([2000, 2024])].pivot(index="geo", columns="year", values="gdp_pc")
print("Regions with 2000:", pivot[2000].notna().sum())
print("Regions with 2024:", pivot[2024].notna().sum())
print("Regions with BOTH 2000 and 2024:", pivot.dropna().shape[0])

dropped = sorted(set(pivot.index) - set(pivot.dropna().index))
print("Dropped because missing 2000 or 2024:", dropped)

GDP PPS regions in file: 244
Regions with 2000: 232
Regions with 2024: 244
Regions with BOTH 2000 and 2024: 232
Dropped because missing 2000 or 2024: ['BE10', 'BE21', 'BE22', 'BE23', 'BE24', 'BE25', 'BE31', 'BE32', 'BE33', 'BE34', 'BE35', 'FRY5']
